# OpenVLA в симуляторе LIBERO: замкнутый цикл + физика, которую модель не видит

<a href="https://colab.research.google.com/github/bespatim/vla-manipulation-research/blob/experiments/libero-colab/notebooks/libero_openvla_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Что делает ноутбук:
1. Создаёт отдельное окружение **Python 3.10** с версиями, как у авторов OpenVLA (PyTorch 2.2.0, transformers 4.40.1),
   и ставит симулятор **LIBERO**.
2. Загружает **OpenVLA, дообученную авторами** на задачах LIBERO (Приложение E статьи OpenVLA).
3. Запускает модель в **замкнутом цикле**: кадр → модель → действие → физика → новый кадр.
4. На каждом шаге записывает то, чего модель **не получает**: силу контакта пальцев с предметами,
   положение предметов, суставы, камеру на запястье.
5. Сравнивает процент успеха с результатом авторов (Таблица 12: Spatial 84.7%, Object 88.4%, Goal 79.2%, Long 53.7%).

**Перед запуском:** меню **Runtime → Change runtime type → T4 GPU → Save**.
Затем запускайте блоки кода по очереди кнопкой ▶ слева от блока (или Shift+Enter).

| Блок | Время |
|---|---|
| Установка | ~5–7 мин |
| Превью сцены | ~1 мин |
| Первый эпизод (с загрузкой модели ~15 ГБ) | ~10–15 мин |

## 1. Проверка видеокарты

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Установка (~5–7 мин)

Colab работает на Python 3.13, а код OpenVLA требует `tokenizers==0.19.1`, которой для 3.13 нет.
Поэтому модель и симулятор запускаются в отдельном окружении Python 3.10 (`/content/venv310`),
а графики и видео показываются здесь, в ноутбуке.

In [ ]:
BRANCH = "experiments/libero-colab"   # после слияния в main поменять на "main"
PY = "/content/venv310/bin/python"
REPO_DIR = "/content/vla-manipulation-research"

%cd /content
!test -d vla-manipulation-research || git clone -q -b {BRANCH} https://github.com/bespatim/vla-manipulation-research.git
!cd vla-manipulation-research && git pull -q
!test -d LIBERO || git clone -q --depth 1 https://github.com/Lifelong-Robot-Learning/LIBERO.git
# Новые версии PyTorch по умолчанию не загружают numpy-массивы через torch.load — разрешаем явно
!sed -i 's/init_states = torch.load(init_states_path)/init_states = torch.load(init_states_path, weights_only=False)/' LIBERO/libero/libero/benchmark/__init__.py

!pip install -q uv
!uv venv -q --python 3.10 /content/venv310
!uv pip install -q --python {PY} -r {REPO_DIR}/notebooks/requirements-libero-colab.txt
!uv pip install -q --python {PY} --no-deps "bddl==1.0.1"
!{PY} -c "import sys, torch, transformers; print('Python', sys.version.split()[0], '| torch', torch.__version__, '| transformers', transformers.__version__, '| CUDA', torch.cuda.is_available())"

## 3. Симулятор без модели

Проверяем, что LIBERO рисует картинку на видеокарте (режим EGL — без монитора).
Слева — сырой кадр внешней камеры, в центре — что получает модель, справа — камера на запястье (модель её не видит).

In [ ]:
import os, sys

os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
icd = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(icd):
    os.makedirs(os.path.dirname(icd), exist_ok=True)
    with open(icd, "w") as f:
        f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}')

SUITE = "libero_spatial"   # libero_spatial | libero_object | libero_goal | libero_10
SCRIPT = f"{REPO_DIR}/scripts/libero_openvla_eval.py"
OUT = f"/content/results/{SUITE}"

!{PY} {SCRIPT} --libero-root /content/LIBERO --suite {SUITE} --tasks 0 --preview /content/preview.png

from IPython.display import Image, Video, display
display(Image("/content/preview.png"))

## 4. Первый эпизод (~10–15 мин, из них большая часть — скачивание модели)

На T4 (15 ГБ) модель грузится в **4-битном** виде (~7 ГБ). По данным авторов, 4 бита не снижают качество
(статья, Таблица 11). На A100 автоматически включится полная точность bf16, как у авторов.

Видео: **слева** — что видит модель, **справа** — камера на запястье. Скорость видео = реальное время
симуляции (20 шагов/с).

In [ ]:
!{PY} {SCRIPT} --libero-root /content/LIBERO --suite {SUITE} --tasks 0 --trials 1 --out {OUT}/first
display(Video(f"{OUT}/first/{SUITE}_task00_trial00.mp4", embed=True, width=768))

## 5. Серия эпизодов и сравнение с авторами

`TASKS` — номера задач (0–9), `TRIALS` — попыток на задачу. 10 задач × 3 попытки = 30 эпизодов ≈ 1–1,5 часа.
Результаты пишутся по ходу, так что при обрыве сессии сделанное не пропадёт (если сохранить на Диск, шаг 7).

In [ ]:
TASKS = " ".join(str(t) for t in range(10))
TRIALS = 3

!{PY} {SCRIPT} --libero-root /content/LIBERO --suite {SUITE} --tasks {TASKS} --trials {TRIALS} --out {OUT}/series

In [ ]:
import numpy as np
import pandas as pd

sys.path.insert(0, f"{REPO_DIR}/scripts")
import libero_openvla_eval as E

def wilson(k, n, z=1.96):
    # 95% доверительный интервал для доли успехов (Уилсон)
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return 100 * (c - h), 100 * (c + h)

df = pd.read_csv(f"{OUT}/series/summary.csv")
k, n = int(df.success.sum()), len(df)
lo, hi = wilson(k, n)
print(f"Мой результат: {100 * k / n:.1f}% ({k}/{n}), 95% ДИ [{lo:.0f}%, {hi:.0f}%]")
print(f"У авторов:     {E.PAPER_SUCCESS[SUITE]}% (1500 попыток, bf16, A100)")
display(df.groupby("instruction").success.agg(["mean", "count"]).rename(columns={"mean": "успех", "count": "попыток"}))

## 6. Что происходило внутри эпизода

Для одного эпизода: высота захвата и целевого предмета, раскрытие пальцев и **сила контакта пальцев
с предметом** — сигнал, которого у модели нет.

In [ ]:
import glob
import matplotlib.pyplot as plt

def plot_episode(csv_path):
    log = pd.read_csv(csv_path)
    objs = [c[len("force_"):] for c in log.columns if c.startswith("force_") and c != "force_other"]
    fig, ax = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
    ax[0].plot(log.step, log.eef_z, label="захват")
    for o in objs:
        if f"{o}_z" in log:
            ax[0].plot(log.step, log[f"{o}_z"], label=o)
    ax[0].set_ylabel("высота, м"); ax[0].legend(fontsize=8)
    ax[1].plot(log.step, log.gripper_q0 - log.gripper_q1, label="раскрытие пальцев")
    ax[1].plot(log.step, (log.act_env_6 > 0) * 0.08, "--", label="команда: закрыть")
    ax[1].set_ylabel("м"); ax[1].legend(fontsize=8)
    for o in objs:
        ax[2].plot(log.step, log[f"force_{o}"], label=o)
    ax[2].plot(log.step, log.force_other, ":", label="прочее (стол и т.п.)")
    ax[2].set_ylabel("сила контакта, Н"); ax[2].set_xlabel("шаг (20 Гц)"); ax[2].legend(fontsize=8)
    fig.suptitle(csv_path.split("/")[-1])
    plt.tight_layout(); plt.show()

episodes = sorted(glob.glob(f"{OUT}/series/*.csv"))
episodes = [e for e in episodes if not e.endswith("summary.csv")]
failed = [f"{OUT}/series/{SUITE}_task{t:02d}_trial{i:02d}.csv" for t, i in df.loc[~df.success, ["task_id", "trial"]].values]
print("неудачные эпизоды:", [f.split("/")[-1] for f in failed])
plot_episode(failed[0] if failed else episodes[0])

## 7. Сохранение результатов

Сессия Colab стирается при закрытии. Сохраните результаты на Google Диск или скачайте архив
и положите в репозиторий в `results/`.

In [ ]:
# На Google Диск:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/vla-results && cp -r /content/results/* /content/drive/MyDrive/vla-results/

In [ ]:
# Или архивом на компьютер:
!cd /content && zip -qr results.zip results
from google.colab import files
files.download("/content/results.zip")